# 🎧 تبدیل صدای کلاس (فارسی) به متن — Whisper large-v3

این نوت‌بوک فایل صوتیِ دوساعته‌ی فارسی را با بهترین کیفیت به متن تبدیل می‌کند.
خروجی، دو فایلِ متنی است که داخل Google Drive خودت ذخیره می‌شود:
- یک متنِ ساده (`...txt`)
- یک متنِ همراه با زمان‌بندی (`..._timestamps.txt`)

### قبل از شروع — حتماً GPU را روشن کن:
از منوی بالا: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

بعد سلول‌ها را به‌ترتیب از بالا به پایین اجرا کن (دکمه‌ی ▶ کنار هر سلول).


## ۱) نصب کتابخانه و بررسی GPU


In [ ]:
!pip -q install faster-whisper
import torch
print("GPU در دسترس است؟ ", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ اگر False است: Runtime → Change runtime type → T4 GPU را انتخاب کن و این سلول را دوباره اجرا کن.")


## ۲) وصل‌کردن Google Drive
بعد از اجرا، یک پنجره باز می‌شود؛ اجازه‌ی دسترسی بده. فایل‌های درایوت در مسیر `/content/drive/MyDrive/` می‌آیند.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## ۳) مسیر فایل صوتی را وارد کن و رونویسی را اجرا کن
از پنلِ سمتِ چپ (آیکن پوشه 📁) فایلِ صوتی‌ات را در `drive/MyDrive` پیدا کن، رویش راست‌کلیک → **Copy path**، و در خط `AUDIO_PATH` بچسبان.

پشتیبانی از فرمت‌ها: `m4a, mp3, wav, ogg, opus, mp4` و ... .


In [ ]:
# ↓↓↓ فقط این دو خط را تنظیم کن ↓↓↓
AUDIO_PATH = "/content/drive/MyDrive/your-audio-file.m4a"   # ← مسیر فایل صوتی‌ات
OUTPUT_DIR = "/content/drive/MyDrive/transcripts"           # ← خروجی اینجا ذخیره می‌شود
# ↑↑↑ -------------------------- ↑↑↑

import os, torch
from faster_whisper import WhisperModel

assert os.path.exists(AUDIO_PATH), f'فایل پیدا نشد: {AUDIO_PATH}  (مسیر را درست بچسبان)'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"بارگذاری مدل large-v3 روی {device} ... (بار اول ~۳ گیگ دانلود می‌کند، چند دقیقه صبر کن)")
model = WhisperModel("large-v3", device=device, compute_type=compute_type)

print("شروع رونویسی... ۲ ساعت روی GPU حدوداً ۱۰ تا ۲۵ دقیقه طول می‌کشد.")
segments, info = model.transcribe(
    AUDIO_PATH, language="fa", beam_size=5, vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
)

total = info.duration or 0
base = os.path.splitext(os.path.basename(AUDIO_PATH))[0]
txt_path = os.path.join(OUTPUT_DIR, base + ".txt")
ts_path  = os.path.join(OUTPUT_DIR, base + "_timestamps.txt")

def hms(s):
    h=int(s//3600); m=int((s%3600)//60); sec=int(s%60)
    return f"{h:02d}:{m:02d}:{sec:02d}"

with open(txt_path,"w",encoding="utf-8") as f1, open(ts_path,"w",encoding="utf-8") as f2:
    for seg in segments:
        line = seg.text.strip()
        f1.write(line + "\n")
        f2.write(f"[{hms(seg.start)} → {hms(seg.end)}] {line}\n")
        pct = (seg.end/total*100) if total else 0
        print(f"\r{pct:5.1f}%  تا {hms(seg.end)}", end="")

print("\n✅ تمام شد!")
print("متن ساده:      ", txt_path)
print("متن با زمان‌بندی:", ts_path)


## ۴) پیش‌نمایش متن


In [ ]:
print(open(txt_path, encoding="utf-8").read()[:2500])


## ۵) تمام!
فایل‌های متنی الان داخلِ پوشه‌ی **`transcripts`** در Google Drive تو هستند.

حالا همان پوشه/فایل را با من به‌اشتراک بگذار (یا بگو در درایو هست) تا بخوانمش و از روی درسِ معلمت **حل‌های قدم‌به‌قدم و سؤال‌های تمرینیِ جدید** برای نرم‌افزار بسازم. 🎯
